# 🔬 NER Diagnostic: Span Analysis & Improvement Guide

This notebook loads the saved ensemble checkpoints and runs a **deep diagnostic** on `labeling_sample.csv`.

**What you get:**
- Side-by-side raw span → cleaned span → gold comparison
- False Positive catalogue (what we predict but shouldn't)
- False Negative catalogue (what we miss and why)
- Per-token confidence histogram (where the model is uncertain)
- Threshold sweep curve on real data
- Actionable improvement checklist printed at the end

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pytorch-lightning wandb

In [ ]:
# ── Cell 2: Imports & src path ─────────────────────────────────────────────────
import os, sys, ast, re, string, glob
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

# Unzip data + sync src
DATA_ZIP   = '/content/drive/MyDrive/DL/Assignment-3/kse-ua-location-extraction-2026.zip'
SRC_DIR    = '/content/drive/MyDrive/DL/Assignment-3/src'
CKPT_DIR   = '/content/drive/MyDrive/DL/Assignment-3/checkpoints'

!cp "{DATA_ZIP}" /content/
!unzip -q -o /content/kse-ua-location-extraction-2026.zip -d /content/
if os.path.exists(SRC_DIR):
    !cp -r "{SRC_DIR}" /content/

if '/content' not in sys.path:
    sys.path.insert(0, '/content')

from src.preprocessing import (
    hard_processing, comp_metric, comp_metric_detailed,
    WordTokenizer, WordVocab, CharVocab,
    clean_entity_boundaries, extract_entities_from_probs, parse_locations_list
)
from src.models import RecurrentNERLightningModule
from src.dataset import create_dataloaders

print('✅ Imports OK')

In [ ]:
# ── Cell 3: Config — EDIT these to match your last training run ────────────────
class Config:
    DATA_DIR        = '/content'
    CHECKPOINT_DIR  = '/content/drive/MyDrive/DL/Assignment-3/checkpoints'
    NUM_CLASSES     = 7
    MAX_LEN         = 128
    MAX_CHAR_LEN    = 16
    VAL_BATCH_SIZE  = 32
    # Weights used during training (must match checkpoint)
    CLASS_WEIGHTS   = [1.0, 15.0, 15.0, 5.0, 5.0, 5.0, 5.0]
    # Glob pattern — picks all fold checkpoints automatically
    CKPT_GLOB       = 'f[0-9]_*.ckpt'
    # Threshold range to sweep on labeling_sample
    TH_RANGE        = [round(x, 2) for x in np.arange(0.30, 0.96, 0.05)]

# ── Load labeling_sample ───────────────────────────────────────────────────────
df_sample = pd.read_csv(os.path.join(Config.DATA_DIR, 'labeling_sample.csv'))
df_sample['gold'] = df_sample['locations'].apply(parse_locations_list)
print(f'labeling_sample: {len(df_sample)} rows, {df_sample["gold"].apply(len).sum()} total gold entities')

# ── Build vocab from training data (same 400k sample as training) ──────────────
print('Building vocab...')
df_uk = pd.read_csv(os.path.join(Config.DATA_DIR, 'uk_geo_dataset.csv'), nrows=300000)
df_ru = pd.read_csv(os.path.join(Config.DATA_DIR, 'ru_geo_dataset.csv'), nrows=100000)
df_all = pd.concat([df_uk, df_ru], ignore_index=True)

tokenizer = WordTokenizer(lowercase=False)
vocab     = WordVocab()
char_vocab = CharVocab()

all_tokens = [tokenizer.tokenize(str(t)) for t in df_all['text'].fillna('')]
vocab.build_vocab(all_tokens, min_freq=2, max_size=60000)
print(f'Vocab size: {len(vocab)} | Char vocab: {len(char_vocab)}')

In [ ]:
# ── Cell 4: Load all fold checkpoints ─────────────────────────────────────────
ckpt_paths = sorted(glob.glob(os.path.join(Config.CHECKPOINT_DIR, Config.CKPT_GLOB)))
print(f'Found {len(ckpt_paths)} checkpoints:')
for p in ckpt_paths:
    print(' ', os.path.basename(p))

models = []
for p in ckpt_paths:
    m = RecurrentNERLightningModule.load_from_checkpoint(
        p, class_weights=Config.CLASS_WEIGHTS
    ).eval().cuda()
    models.append(m)
    print(f'  Loaded: {os.path.basename(p)}')

print(f'\n✅ {len(models)} models ready')

In [ ]:
# ── Cell 5: Run ensemble inference on labeling_sample ─────────────────────────
# Returns per-token LOC probability and raw span data for every sample

_, _, _, sample_loader = create_dataloaders(
    train_df=df_sample,  # dummy — we just need sample_loader
    val_df=df_sample,
    test_df=df_sample,
    vocab=vocab,
    char_vocab=char_vocab,
    tokenizer=tokenizer,
    batch_size=Config.VAL_BATCH_SIZE,
    val_batch_size=Config.VAL_BATCH_SIZE,
    max_len=Config.MAX_LEN,
)

all_texts, all_offsets, all_gold_locs, all_lengths = [], [], [], []
all_ensemble_probs   = []   # List[np.array shape (seq_len, 7)]
all_loc_probs        = []   # List[np.array shape (seq_len,)]  — B-LOC + I-LOC
all_tokens_per_sample = []  # List[List[str]]

with torch.no_grad():
    for batch in sample_loader:
        input_ids = batch['input_ids'].cuda()
        lengths   = batch['lengths'].cuda()
        char_ids  = batch['char_ids'].cuda()

        # Soft-vote across folds
        ens_probs = torch.zeros(input_ids.size(0), input_ids.size(1), Config.NUM_CLASSES, device='cuda')
        for m in models:
            logits = m(input_ids, lengths=lengths, char_ids=char_ids)
            ens_probs += F.softmax(logits, dim=-1)
        ens_probs = (ens_probs / len(models)).cpu().numpy()  # (B, S, 7)

        for i, text in enumerate(batch['texts']):
            L = lengths[i].item()
            all_texts.append(text)
            all_offsets.append(batch['offsets'][i][:L])
            all_gold_locs.append(batch['gold_locations'][i])
            all_lengths.append(L)
            ep = ens_probs[i][:L]        # (L, 7)
            all_ensemble_probs.append(ep)
            # Combined LOC prob = P(B-LOC) + P(I-LOC)
            all_loc_probs.append(ep[:, 1] + ep[:, 2])  # (L,)
            # Reconstruct tokens from offsets
            all_tokens_per_sample.append([text[s:e] for s, e in batch['offsets'][i][:L]])

print(f'✅ Inference done on {len(all_texts)} samples')

In [ ]:
# ── Cell 6: Threshold Sweep Curve ─────────────────────────────────────────────
print('=' * 65)
print(f'{"Threshold":>12} | {"Precision":>10} | {"Recall":>8} | {"F1":>8} | {"TP":>5} | {"FP":>5} | {"FN":>5}')
print('-' * 65)

best_f1, best_th = 0.0, 0.5
th_results = []

for th in Config.TH_RANGE:
    y_pred = [
        list(dict.fromkeys(
            extract_entities_from_probs(text, offsets, probs, threshold=th)
        ))
        for text, offsets, probs in zip(all_texts, all_offsets, all_loc_probs)
    ]
    m = comp_metric_detailed(all_gold_locs, y_pred)
    th_results.append({'th': th, **m})
    marker = ' ◀ BEST' if m['f1'] > best_f1 else ''
    if m['f1'] > best_f1:
        best_f1, best_th = m['f1'], th
    print(f"{th:>12.2f} | {m['precision']:>10.4f} | {m['recall']:>8.4f} | {m['f1']:>8.4f} |"
          f" {int(m['tp']):>5} | {int(m['fp']):>5} | {int(m['fn']):>5}{marker}")

print('=' * 65)
print(f'\n✅ Best threshold = {best_th:.2f} → F1 = {best_f1:.4f}')
print(f'   (Use this as ensemble_th for Kaggle submission!)')

In [ ]:
# ── Cell 7: Per-Sample Span Diagnosis ─────────────────────────────────────────
# Shows: raw predicted spans, cleaned spans, gold — for EVERY sample
# Annotates each prediction as TP / FP / FN

DIAG_TH = best_th   # use the best threshold found above

print(f'Using threshold = {DIAG_TH:.2f}')
print('=' * 90)

fp_catalogue = []   # all false positives across samples
fn_catalogue = []   # all false negatives across samples
tp_catalogue = []   # true positives

for idx, (text, offsets, probs, gold) in enumerate(
    zip(all_texts, all_offsets, all_loc_probs, all_gold_locs)
):
    # Collect RAW spans (before clean_entity_boundaries)
    raw_spans = extract_entities_from_probs(text, offsets, probs, threshold=DIAG_TH, clean_boundaries=False)
    # Cleaned spans (what actually goes into submission)
    cleaned_spans = list(dict.fromkeys(
        extract_entities_from_probs(text, offsets, probs, threshold=DIAG_TH, clean_boundaries=True)
    ))
    cleaned_spans = [s for s in cleaned_spans if s]  # drop empty

    gold_norm   = {hard_processing(g) for g in gold if hard_processing(g)}
    pred_norm   = {hard_processing(p): p for p in cleaned_spans if hard_processing(p)}
    pred_keys   = set(pred_norm.keys())

    tp_keys = pred_keys & gold_norm
    fp_keys = pred_keys - gold_norm
    fn_keys = gold_norm - pred_keys

    for k in tp_keys: tp_catalogue.append({'sample': idx, 'span': pred_norm[k], 'norm': k})
    for k in fp_keys: fp_catalogue.append({'sample': idx, 'span': pred_norm[k], 'norm': k, 'text_snippet': text[:120]})
    for g in gold:
        gn = hard_processing(g)
        if gn in fn_keys:
            fn_catalogue.append({'sample': idx, 'gold': g, 'norm': gn, 'text_snippet': text[:120]})

    if raw_spans or gold:
        print(f'\n[Sample {idx}]')
        print(f'  TEXT  : {repr(text[:150])}')
        for rs, cs in zip(raw_spans, cleaned_spans):
            diff = f'  ← stripped: {repr(rs)}' if rs != cs else ''
            tag  = '✅ TP' if hard_processing(cs) in gold_norm else '❌ FP'
            print(f'  PRED  : {repr(cs):50s} {tag}{diff}')
        for g in gold:
            if hard_processing(g) not in pred_keys:
                print(f'  MISSED: {repr(g):50s} ⚠️  FN')

print('\n' + '=' * 90)

In [ ]:
# ── Cell 8: False Positive Deep-Dive ──────────────────────────────────────────
# What is the model confidently wrong about?
print(f'\n🔴 FALSE POSITIVES ({len(fp_catalogue)} total):')
print('-' * 80)
fp_df = pd.DataFrame(fp_catalogue)
if not fp_df.empty:
    # Most common FP spans
    fp_counts = Counter(fp_df['span'])
    print('Most repeated FP spans:')
    for span, cnt in fp_counts.most_common(15):
        print(f'  {cnt:3d}x  {repr(span)}')

    print('\nAll FP details:')
    for _, row in fp_df.iterrows():
        print(f"  Sample {row['sample']:2d} | {repr(row['span']):40s} | context: {repr(row['text_snippet'][:80])}")
else:
    print('  (none!)')

print(f'\n🟡 FALSE NEGATIVES ({len(fn_catalogue)} total — entities we miss):')
print('-' * 80)
fn_df = pd.DataFrame(fn_catalogue)
if not fn_df.empty:
    fn_counts = Counter(fn_df['gold'])
    print('Most repeatedly missed gold entities:')
    for span, cnt in fn_counts.most_common(15):
        print(f'  {cnt:3d}x  {repr(span)}')

    print('\nAll FN details:')
    for _, row in fn_df.iterrows():
        print(f"  Sample {row['sample']:2d} | GOLD={repr(row['gold']):40s} | context: {repr(row['text_snippet'][:80])}")
else:
    print('  (none!)')

In [ ]:
# ── Cell 9: Token-level Confidence Analysis ────────────────────────────────────
# Shows where the model is 'confused' — tokens near the threshold boundary
# These are the highest-leverage tokens to fix via training

BOUNDARY_LOW  = best_th - 0.10
BOUNDARY_HIGH = best_th + 0.10

uncertain_tokens = []   # tokens the model scores close to threshold
confident_fp_tokens = []  # tokens model is very confident about but are FP

for text, offsets, probs, ep, tokens, gold in zip(
    all_texts, all_offsets, all_loc_probs, all_ensemble_probs, all_tokens_per_sample, all_gold_locs
):
    gold_norm = {hard_processing(g) for g in gold if hard_processing(g)}

    for tok, (s, e), p, full_p in zip(tokens, offsets, probs, ep):
        # Is this token actually inside a gold span?
        in_gold = any(
            any(text[gs:ge] in g or g in text[gs:ge] for g in gold)
            for gs, ge in []
        )  # rough — just use prob threshold comparison below

        if BOUNDARY_LOW <= p <= BOUNDARY_HIGH:
            uncertain_tokens.append({
                'token': tok, 'loc_prob': round(float(p), 3),
                'p_bloc': round(float(full_p[1]), 3),
                'p_iloc': round(float(full_p[2]), 3),
                'p_org':  round(float(full_p[3] + full_p[4]), 3),
                'p_per':  round(float(full_p[5] + full_p[6]), 3),
                'context': text[max(0, s-30):e+30],
            })
        if p > best_th + 0.15:  # very confident
            confident_fp_tokens.append({
                'token': tok, 'loc_prob': round(float(p), 3),
                'p_org': round(float(full_p[3] + full_p[4]), 3),
                'context': text[max(0, s-30):e+30],
            })

print(f'Tokens near threshold boundary ({BOUNDARY_LOW:.2f}-{BOUNDARY_HIGH:.2f}): {len(uncertain_tokens)}')
print('These are the tokens the model is most unsure about — fixing them has the highest impact on F1:\n')
for t in sorted(uncertain_tokens, key=lambda x: x['loc_prob'], reverse=True)[:25]:
    print(f"  {t['token']:20s} loc={t['loc_prob']:.3f}  B-LOC={t['p_bloc']:.3f}  I-LOC={t['p_iloc']:.3f}  "
          f"ORG={t['p_org']:.3f}  PER={t['p_per']:.3f}")
    print(f"    context: {repr(t['context'])}")

In [ ]:
# ── Cell 10: Boundary Cleaning Audit ──────────────────────────────────────────
# Shows exactly what clean_entity_boundaries() is stripping and whether it helps

print('BOUNDARY CLEANING AUDIT')
print('=' * 90)
print(f'{"RAW SPAN":50s} | {"CLEANED":40s} | CHANGED?')
print('-' * 90)

changed_count = 0
for text, offsets, probs, gold in zip(all_texts, all_offsets, all_loc_probs, all_gold_locs):
    raw_spans     = extract_entities_from_probs(text, offsets, probs, threshold=best_th, clean_boundaries=False)
    cleaned_spans = extract_entities_from_probs(text, offsets, probs, threshold=best_th, clean_boundaries=True)
    for rs, cs in zip(raw_spans, cleaned_spans):
        changed = rs != cs
        if changed:
            changed_count += 1
        marker = '✏️ ' if changed else ''
        print(f'{repr(rs):50s} | {repr(cs):40s} | {marker}')

print(f'\n{changed_count} spans were modified by clean_entity_boundaries()')

In [ ]:
# ── Cell 11: Per-Class Probability Distribution ────────────────────────────────
# Tells you if the 7-class training is working: ORG/PER tokens should have high
# P(ORG)/P(PER) and low P(LOC). LOC tokens should have high P(LOC).

print('PER-CLASS PROBABILITY DISTRIBUTION (across all tokens in labeling_sample)\n')
print(f'{"class":12s} | {"mean prob":>10} | {"median":>8} | {"p90":>8} | {"p99":>8}')
print('-' * 55)

class_names = ['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER']
all_probs_flat = np.concatenate([ep for ep in all_ensemble_probs], axis=0)  # (N_tokens, 7)

for ci, cname in enumerate(class_names):
    col = all_probs_flat[:, ci]
    print(f'{cname:12s} | {col.mean():>10.4f} | {np.median(col):>8.4f} | {np.percentile(col, 90):>8.4f} | {np.percentile(col, 99):>8.4f}')

print()
# Confusion: tokens where model gives high LOC but higher ORG (misassigned)
loc_prob_col  = all_probs_flat[:, 1] + all_probs_flat[:, 2]
org_prob_col  = all_probs_flat[:, 3] + all_probs_flat[:, 4]
confused_mask = (loc_prob_col > 0.2) & (org_prob_col > loc_prob_col * 0.5)
print(f'Tokens where ORG prob is >50% of LOC prob (potential LOC/ORG confusion): {confused_mask.sum()}')

In [ ]:
# ── Cell 12: Actionable Improvement Report ─────────────────────────────────────
print('=' * 90)
print('📊 DIAGNOSTIC SUMMARY & IMPROVEMENT ROADMAP')
print('=' * 90)

# Re-compute final metrics at best threshold
y_pred_best = [
    list(dict.fromkeys(
        extract_entities_from_probs(text, offsets, probs, threshold=best_th)
    ))
    for text, offsets, probs in zip(all_texts, all_offsets, all_loc_probs)
]
m_best = comp_metric_detailed(all_gold_locs, y_pred_best)

total_gold = sum(len(g) for g in all_gold_locs)
print(f'\n📌 Ensemble on labeling_sample @ threshold={best_th:.2f}:')
print(f'   Precision = {m_best["precision"]:.4f} | Recall = {m_best["recall"]:.4f} | F1 = {m_best["f1"]:.4f}')
print(f'   TP={int(m_best["tp"])}  FP={int(m_best["fp"])}  FN={int(m_best["fn"])}  Total gold={total_gold}')

print()
precision_limited = m_best['precision'] < m_best['recall']
recall_limited    = m_best['recall'] < m_best['precision']

if recall_limited:
    print('⚠️  RECALL-LIMITED model (we miss more than we hallucinate).')
    print('   → Lower threshold, or train with higher class weight on LOC.')
    print('   → Check FN catalogue above: are FNs short 1-word locations? Use smaller max_char_len.')
    print('   → Consider adding more positive training examples (more RU data).')
elif precision_limited:
    print('⚠️  PRECISION-LIMITED model (we hallucinate more than we miss).')
    print('   → Raise threshold or increase ORG/PER class weights to push org-names away from LOC.')
    print('   → Check FP catalogue: are FPs org names (ТОВ, КП, МВФ)? Increase w_org to 10.0.')

print()
print('=' * 90)
print('🔧 RANKED IMPROVEMENT ACTIONS (highest expected Kaggle gain first):')
print('=' * 90)
print()

actions = [
    ('1', 'USE THIS BEST THRESHOLD for Kaggle submission',
     f'Set ensemble_th = {best_th:.2f} in notebook Cell 7 inference.'
     ' Previous run used 0.95 (synthetic-calibrated) — wrong for real data!'),

    ('2', 'Class weight tuning for ORG/PER',
     'If FP catalogue shows org names (ТОВ, МВФ, Укрзалізниця etc.) → raise ORG/PER weights to 8-10.\n'
     '     Current: CLASS_WEIGHTS = [1, 15, 15, 5, 5, 5, 5]\n'
     '     Try:     CLASS_WEIGHTS = [1, 15, 15, 8, 8, 8, 8]  or [1, 20, 20, 5, 5, 5, 5]'),

    ('3', 'More training epochs / folds',
     'Currently 5 epochs × 3 folds. Try 8 epochs × 5 folds.\n'
     '     More folds = better calibration on small labeling_sample benchmark.\n'
     '     More epochs = model sees more of the Ukrainian morphology patterns.'),

    ('4', 'Increase Russian data ratio',
     'Current: 300k UA + 100k RU. Test/labeling_sample is Kyiv Telegram (mixed UA/RU).\n'
     '     Try: 300k UA + 200k RU to improve generalisation to Russian-language location names.'),

    ('5', 'BiGRU ensemble member for model diversity',
     'Add a BiGRU (rnn_type="GRU") 3-fold model and blend probabilities with the BiLSTM ensemble.\n'
     '     Soft-vote across 6 models (3 LSTM + 3 GRU) = lower variance, higher F1.'),

    ('6', 'CRF decoding layer instead of threshold',
     'Replace probability threshold with a CRF layer on top of BiLSTM.\n'
     '     CRF enforces BIO constraints (I-LOC can only follow B-LOC) — reduces\'fragmented\' spans.\n'
     '     Still fully recurrent — no transformers — so compliant with constraints.\n'
     '     Library: torchcrf (pip install pytorch-crf)'),

    ('7', 'Hidden size / depth increase',
     'Current: hidden_dim=256, num_layers=2. Try hidden_dim=512 or num_layers=3.\n'
     '     May help with long compound location names like "Шулявський шляхопровід".'),

    ('8', 'Gaussian noise regularisation on embeddings',
     'Add small Gaussian noise to word embeddings during training (std=0.1).\n'
     '     Improves robustness to OOV tokens common in Telegram text (typos, emojis in-word).'),
]

for num, title, detail in actions:
    print(f'  [{num}] {title}')
    for line in detail.split('\n'):
        print(f'      {line}')
    print()

print('=' * 90)
print('💾 Save this output! Paste the FP/FN catalogues + threshold = {} into CONTEXT_HANDOVER.md'.format(best_th))
print('   before the next training run.')